# TT kullanım teşhisi — kayıtlı 100 oyun

Bu notebook yeni maç oynatmaz. Son 100 maçın kontrol tarafındaki pozisyonlarını
kullanarak TT sorgularını, geçmiş/derinlik retlerini, gerçek kesmeleri ve tablo
temizlemelerini sayar. TT politikası ve motor kodu değiştirilmez.

Her pozisyonda normal ve sayaçlı motor aynı kayıtlı **düğüm sınırıyla** çalışır.
Hamle/skor/derinlik/düğüm ve TT içeriği farklıysa teşhis durur. Sayaç ek yükü
nedeniyle 100 ms duvar süresiyle hız karşılaştırması yapılmaz; sonuç Elo testi değildir.
TT aynı kayıtlı oyun içinde sıcak kalır, her yeni oyunda sıfırdan başlar.

`codex/tt-diagnostics` dalından bu dosyayı Kaggle'a içe alın. Internet'i açın,
Contents yazma yetkili `GITHUB_TOKEN` secret'ını ekleyin ve Run All çalıştırın.
Tüm gerçek aramalar ve tam testler yalnız Kaggle'dadır.


In [ ]:
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "codex/tt-diagnostics"
WORKSPACE = "/kaggle/working/acemi-tt"
SESSION_ID = "tt-usage-node-replay-v1"
SOURCE_SESSION_ID = "time-policy-100vs150-100ms-100games-v1"
RETRY_INCOMPLETE = False  # Only after stopping an interrupted old Kaggle session.


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("TT diagnostics run only in Kaggle.")

def checked(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {args[0]}\n{(result.stdout + result.stderr)[-4000:]}")
    return result.stdout.strip()

workspace = Path(WORKSPACE)
if not workspace.exists():
    checked(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(workspace)])
else:
    checked(["git", "-C", str(workspace), "fetch", "origin", GITHUB_BRANCH])
    local = subprocess.run(["git", "-C", str(workspace), "show-ref", "--verify", "--quiet", f"refs/heads/{GITHUB_BRANCH}"])
    if local.returncode == 0:
        checked(["git", "-C", str(workspace), "switch", GITHUB_BRANCH])
    else:
        checked(["git", "-C", str(workspace), "switch", "--create", GITHUB_BRANCH, "--track", f"origin/{GITHUB_BRANCH}"])
    checked(["git", "-C", str(workspace), "merge", "--ff-only", f"origin/{GITHUB_BRANCH}"])
os.chdir(workspace)
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))
if not (workspace / "tools" / "tt_diagnostics.py").exists():
    raise RuntimeError("Import the latest notebook from codex/tt-diagnostics and rerun setup.")
checked([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"], cwd=workspace)

session = workspace / "data" / "tt" / SESSION_ID
print("TT diagnostic output:", session)
print("Source: existing 100-game control positions; no new matches.")


In [ ]:
# Publish records only. The token is never written to a file or a remote URL.
import base64
from kaggle_secrets import UserSecretsClient

def publish_results(message):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Add GITHUB_TOKEN under Kaggle Add-ons / Secrets.")
    auth = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": auth, "GIT_TERMINAL_PROMPT": "0"})
    def git(*args, check=True):
        result = subprocess.run(["git", *args], cwd=workspace, env=env, text=True, capture_output=True)
        if check and result.returncode:
            details = (result.stdout + result.stderr).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
            raise RuntimeError(f"Git {args[0]} failed: {details[-2000:]}")
        return result
    if git("branch", "--show-current").stdout.strip() != GITHUB_BRANCH:
        raise RuntimeError("Wrong results branch; rerun repository setup.")
    git("config", "user.name", "Acemi Kaggle A-B")
    git("config", "user.email", "acemi-kaggle-runner@users.noreply.github.com")
    paths = [str(session.relative_to(workspace))]
    git("add", "--", *paths)
    if git("diff", "--cached", "--quiet", check=False).returncode:
        git("commit", "-m", message)
    git("fetch", "origin", GITHUB_BRANCH)
    if int(git("rev-list", "--count", "HEAD..FETCH_HEAD").stdout.strip()):
        git("rebase", "FETCH_HEAD")
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    print("Published:", message)


In [ ]:
# Prepare immutable inputs, or verify the saved protocol on resume.
if (session / "protocol.json").exists():
    command = [sys.executable, "-u", "-m", "tools.tt_diagnostics", "verify", "--session", str(session)]
else:
    command = [sys.executable, "-u", "-m", "tools.tt_diagnostics", "prepare", "--id", SESSION_ID,
               "--source-session", SOURCE_SESSION_ID]
result = subprocess.run(command, cwd=workspace)
if result.returncode:
    raise RuntimeError("TT protocol preparation/verification failed. No replay started.")
plan = json.loads((session / "protocol.json").read_text())
if plan["source_session"] != SOURCE_SESSION_ID or plan["games"] != 100:
    raise RuntimeError("Source settings differ; use a new SESSION_ID.")
publish_results(f"data: prepare TT usage diagnostics ({SESSION_ID})")


In [ ]:
# Full correctness tests, then paired normal/instrumented node-limited replay.
command = [sys.executable, "-u", "-m", "tools.tt_diagnostics", "run", "--session", str(session)]
if RETRY_INCOMPLETE:
    command.append("--retry-incomplete")
print("Starting TT diagnostics for 100 recorded games. This does not play new matches.", flush=True)
result = subprocess.run(command, cwd=workspace)
try:
    publish_results(f"data: TT usage diagnostic records ({SESSION_ID})")
except Exception:
    print("Push failed. Files remain in Kaggle; rerun the final cell to publish without searches.")
    raise
if result.returncode:
    raise RuntimeError("TT diagnostics stopped; inspect tests.log / validation.json / error.log. Records were retained.")
print((session / "SUMMARY.md").read_text())


In [ ]:
# Publication only: safe to repeat after push failure. Never starts a search.
publish_results(f"data: finalize TT usage records ({SESSION_ID})")
if (session / "SUMMARY.md").exists():
    print((session / "SUMMARY.md").read_text())
